In [ ]:
import sys

print("Python Version:", sys.version)
# displays the vesion of the python

print("Pyhton Location:" , sys.executable)
# displays the execution location of the python

In [ ]:
# Project information

project_name = "Privacy-Aware Data Engineering Pipeline"

project_objective = """
Build a data engineering pipeline that:

1. Generates or ingests source data.
2. Profiles the data.
3. Identifies data quality issues.
4. Validates records.
5. Quarantines invalid records.
6. Masks or tokenizes sensitive information.
7. Minimizes the analytical dataset.
8. Publishes the final dataset.
9. Maintains an audit trail.
10. Tests and reconciles the results.
"""

print("Project:", project_name)
print("\nProject Objective:")
print(project_objective)

In [ ]:
# Define the columns expected in our source datasets.

CUSTOMER_SCHEMA = {
    "customer_id": "int",
    "customer_name": "string",
    "email": "string",
    "phone": "string",
    "address": "string",
    "city": "string",
    "age": "int",
    "created_at": "datetime"
}

TRANSACTION_SCHEMA = {
    "transaction_id": "int",
    "customer_id": "int",
    "transaction_date": "datetime",
    "amount": "float",
    "status": "string"
}

# Display the schemas

print("CUSTOMER SCHEMA")
for column, data_type in CUSTOMER_SCHEMA.items():
    print(f"{column}: {data_type}")

print("\nTRANSACTION SCHEMA")
for column, data_type in TRANSACTION_SCHEMA.items():
    print(f"{column}: {data_type}")

# This are the possible and expected schemas in the entities

In [ ]:
# Import the libraries that required for data generation.

import pandas as pd
import random
from faker import Faker

# Create a Faker object to generate synthetical/realistic data
fake = Faker()

# Set fixed seeds to make our generated data reproducible.
# Reproducible means we can generate the same sample again.
Faker.seed(42)
random.seed(42)

NUMBER_OF_CUSTOMERS = 100
NUMBER_OF_TRANSACTIONS = 1000
# This defines how many records we want initially

print("Libraries imported successfully!")
print("Faker initialized successfully!")
print("Customers to generate:", NUMBER_OF_CUSTOMERS)
print("Transactions to generate:", NUMBER_OF_TRANSACTIONS)

In [ ]:
# creates a list to store customer records.
customers = []

# generate customer records one by one.
for i in range(NUMBER_OF_CUSTOMERS):

    # generate a unique customer ID.
    customer_id = i + 1

    # generate realistic customer information.
    customer_name = fake.name()
    email = fake.email()
    phone = fake.numerify(text="##########")
    address = fake.address().replace("\n", ", ")
    city = fake.city()

    # generate customer age between 18 and 70.
    age = random.randint(18, 70)

    # generate customer registration date.
    created_at = fake.date_time_between(
        start_date="-3y",
        end_date="now"
    )

    # store all customer details in a dictionary.
    customer = {
        "customer_id": customer_id,
        "customer_name": customer_name,
        "email": email,
        "phone": phone,
        "address": address,
        "city": city,
        "age": age,
        "created_at": created_at
    }

    # add this customer to our list.
    customers.append(customer)

# convert the list of dictionaries into a table/dataframe
customers_df = pd.DataFrame(customers)

customers_df.head()

In [ ]:
# checks how many rows and columns are existed.
print("Dataset shape:", customers_df.shape)

# checks the column names.
print("\nColumn names:")
print(customers_df.columns.tolist())

# checks the data types.
print("\nData types:")
print(customers_df.dtypes)

In [ ]:

# create an empty list to store transaction records.
transactions = []

# get all existing customer IDs because every transaction initially belong to a customer
customer_ids = customers_df["customer_id"].tolist()

# generate the required number of transactions.
for i in range(NUMBER_OF_TRANSACTIONS):

    # generate a unique transaction ID.
    transaction_id = 1001 + i

    # randomly select an existing customer.
    customer_id = random.choice(customer_ids)

    # generate transaction date.
    transaction_date = fake.date_time_between(
        start_date="-1y",
        end_date="now"
    )

    # generate transaction amount.
    amount = round(random.uniform(100, 10000), 2)

    # generate transaction status.
    status = random.choice([
        "SUCCESS",
        "PENDING",
        "FAILED"
    ])

    # create one transaction record.
    transaction = {
        "transaction_id": transaction_id,
        "customer_id": customer_id,
        "transaction_date": transaction_date,
        "amount": amount,
        "status": status
    }

    # Store the transaction.
    transactions.append(transaction)

# convert transaction records into a DataFrame.
transactions_df = pd.DataFrame(transactions)

# display the first five transactions.
transactions_df.head()


In [ ]:

# checks how many rows and colums are existed
print("Transaction dataset shape:", transactions_df.shape)

# check column names.
print("\nColumns:")
print(transactions_df.columns.tolist())

# check data types.
print("\nData types:")
print(transactions_df.dtypes)

# check how many unique customers made transactions.
print("\nUnique customers with transactions:",
      transactions_df["customer_id"].nunique())


In [ ]:

# get customer IDs from the customer dataset.
valid_customer_ids = set(customers_df["customer_id"])

# get customer IDs referenced by transactions.
transaction_customer_ids = set(transactions_df["customer_id"])

# Find transaction customer IDs that don't exist in customers.
invalid_customer_ids = (
    transaction_customer_ids - valid_customer_ids
)

print("Invalid customer IDs:", invalid_customer_ids)

# Verify referential integrity.
assert len(invalid_customer_ids) == 0, \
    "Some transactions reference customers that do not exist!"

print("Referential integrity check PASSED!")


In [ ]:

# Import Path to handle Windows file paths safely.
from pathlib import Path

# Define the main project directory.
PROJECT_ROOT = Path(r"C:\CandidateWorkspace")

# Define the raw data folder.
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Create the folder if it doesn't already exist.
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:", RAW_DATA_DIR)


In [ ]:

# Save customer records as a CSV file.
customers_df.to_csv(
    RAW_DATA_DIR / "customers.csv",
    index=False
)

print("Customers CSV saved successfully!")


In [ ]:

# Save transaction records as a CSV file.
transactions_df.to_csv(
    RAW_DATA_DIR / "transactions.csv",
    index=False
)

print("Transactions CSV saved successfully!")


In [ ]:

# List all files inside the raw data directory.
for file in RAW_DATA_DIR.iterdir():
    print(file.name, "-", file.stat().st_size, "bytes")


In [ ]:

# read the saved CSV files into DataFrames.

customers_source = pd.read_csv(
    RAW_DATA_DIR / "customers.csv"
)

transactions_source = pd.read_csv(
    RAW_DATA_DIR / "transactions.csv"
)

print("Customer data loaded:", customers_source.shape)
print("Transaction data loaded:", transactions_source.shape)


In [ ]:
def profile_dataframe(df, dataset_name):
    """
    Profile a DataFrame and calculate:
    - Data type
    - Total rows
    - Missing values
    - Missing percentage
    - Unique values
    - Unique percentage
    """

    profile = []

    # Inspect each column individually.
    for column in df.columns:

        # Count missing values.
        null_count = df[column].isnull().sum()

        # Count distinct, non-null values.
        unique_count = df[column].nunique(dropna=True)

        # Calculate percentages safely, including empty datasets.
        total_rows = len(df)

        null_percentage = (
            round(null_count / total_rows * 100, 2)
            if total_rows else 0
        )

        unique_percentage = (
            round(unique_count / total_rows * 100, 2)
            if total_rows else 0
        )

        # Store the results for this column.
        profile.append({
            "dataset": dataset_name,
            "column": column,
            "data_type": str(df[column].dtype),
            "row_count": total_rows,
            "null_count": int(null_count),
            "null_percentage": null_percentage,
            "unique_count": int(unique_count),
            "unique_percentage": unique_percentage
        })

    # Return the profile as a DataFrame.
    return pd.DataFrame(profile)


print("Profiling function created successfully!")

In [ ]:

# Profile customer data.
customer_profile = profile_dataframe(
    customers_source,
    "customers"
)

# Profile transaction data.
transaction_profile = profile_dataframe(
    transactions_source,
    "transactions"
)

# Combine both profiling reports.
combined_profile = pd.concat(
    [customer_profile, transaction_profile],
    ignore_index=True
)

# Display the results.
display(combined_profile)


In [ ]:
# Define sensitive columns in the customer dataset.
SENSITIVE_FIELDS = {
    "customers": [
        "customer_name",
        "email",
        "phone",
        "address"
    ],
    "transactions": []
}

# Display sensitive field configuration.
for dataset, columns in SENSITIVE_FIELDS.items():

    print(f"\nDataset: {dataset}")

    for column in columns:
        print("Sensitive column:", column)


In [ ]:

# Define the reports folder.
REPORTS_DIR = PROJECT_ROOT / "reports"

# Create it if necessary.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Save the combined profile.
combined_profile.to_csv(
    REPORTS_DIR / "data_profile.csv",
    index=False
)

print("Data profiling report saved successfully!")


In [ ]:
# Reload customers to convert the phone into stirng
customers_source = pd.read_csv(
    RAW_DATA_DIR / "customers.csv",
    dtype={"phone": "string"}
)

# Convert customer registration date into datetime.
customers_source["created_at"] = pd.to_datetime(
    customers_source["created_at"],
    errors="coerce"
)

# Reload transactions.
transactions_source = pd.read_csv(
    RAW_DATA_DIR / "transactions.csv"
)

# Convert transaction date into datetime.
transactions_source["transaction_date"] = pd.to_datetime(
    transactions_source["transaction_date"],
    errors="coerce"
)

print("CUSTOMER DATA TYPES")
print(customers_source.dtypes)

print("\nTRANSACTION DATA TYPES")
print(transactions_source.dtypes)

In [ ]:
# create copies of our original datasetsw
# the original source dta remains unchanged

customers_test = customers_source.copy()
transactions_test = transactions_source.copy()

print("Test datasets created!")
print("Customers:", customers_test.shape)
print("Transactions:", transactions_test.shape)

In [ ]:
# Missing customer emails
customers_test.loc[0, "email"] = None

# duplicate customer ID
customers_test.loc[1, "custome_id"] = (
    customers_test.loc[0, "customer_id"]
)

# Introduce an invalid age
customers_test.loc[2, "age"] = 150
print("Customer test issue introduced!")

In [ ]:
# missing transaction amount
transactions_test.loc[0,"amount'"] = None

# Negative transaction amount
transactions_test.loc[1, "amount"] = -500

# duplicate transaction ID
transactions_test.loc[2, "transaction_id"] = (
    transactions_test.loc[1, "transaction_id"]
)

# Referance a customer who dose not existed
transaction_test.loc[3, "customer_id"] = 999999

# unexpected transaction status
transactions_test.loc[4,"status"] = "INVALID"

print("Transaction test issues introduces!")

In [ ]:
def validate_customers(df):
    """
    Validate customer records.

    Returns:
        Valid_records: Records that pass all validation rules
        rejected_records: Records that fail validation, with reasons.
    """
    # Work on a copy so that the input DataFrame is not modified.
    data = df.copy()

    # create a list to storee the rejection reasons for each row
    rejection_reasons = [[] for _ in range(len(data))]

    # Rule1: customer_id must not be missing
    invalid_id = data["customer_id"].isna()
    for i in range(len(data)):
        if invalid_id.iloc[i]:
            rejection_reasons[i].append("Missing customer_id")

    # Rule2: Customer ID must be unique
    duplicate_id = (
        data["customer_id"].notna()
        & data["customer_id"].duplicated(keep = False)
    )

    for i in range(len(data)):
        if duplicate_id.iloc[i]:
            rejection_reasons[i].append("Duplicate customer_id")
        
    # Rule 3: Customer name must not be empty.
    invalid_name = (
        data["customer_name"].isna()
        | data["customer_name"].astype("string").str.strip().eq("").fillna(False)
    )

    for i in range(len(data)):
        if invalid_name.iloc[i]:
            rejection_reasons[i].append("Missing customer_name")

    # Rule 4: Email must be present and have a valid basic format.
    email = data["email"].astype("string").str.strip()

    invalid_email = (
        email.isna()
        | ~email.str.match(
            r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
            na=False
        )
    )

    for i in range(len(data)):
        if invalid_email.iloc[i]:
            rejection_reasons[i].append("Missing or invalid email")

    # Rule 5: Phone must be present.
    invalid_phone = (
        data["phone"].isna()
        | data["phone"].astype("string").str.strip().eq("").fillna(False)
    )

    for i in range(len(data)):
        if invalid_phone.iloc[i]:
            rejection_reasons[i].append("Missing phone")

    # Rule 6: Age must be between 18 and 100.
    invalid_age = (
        data["age"].isna()
        | ~data["age"].between(18, 100)
    )

    for i in range(len(data)):
        if invalid_age.iloc[i]:
            rejection_reasons[i].append("Age must be between 18 and 100")

    # Rule 7: City must not be empty.
    invalid_city = (
        data["city"].isna()
        | data["city"].astype("string").str.strip().eq("").fillna(False)
    )

    for i in range(len(data)):
        if invalid_city.iloc[i]:
            rejection_reasons[i].append("Missing city")

    # Identify rows with one or more validation failures.
    rejected_mask = pd.Series(
        [len(reasons) > 0 for reasons in rejection_reasons],
        index=data.index
    )

    # Separate valid and rejected records.
    valid_records = data.loc[~rejected_mask].copy()
    rejected_records = data.loc[rejected_mask].copy()

    # Add a readable explanation for each rejected record.
    rejected_records["reject_reason"] = [
        "; ".join(rejection_reasons[i])
        for i in range(len(data))
        if len(rejection_reasons[i]) > 0
    ]

    return valid_records, rejected_records


print("Customer validation function created successfully!")

In [ ]:
# Validate the customer test data set

valid_customers, rejected_customers = validate_customers(customers_test)

# Validation reults
print("CUSTOMER VALIDATION RESULTS")
print("---------------------------")

print("Total input records:", len(customers_test))
print("Valid records:", len(valid_customers))
print("Rejected records:", len(rejected_customers))

# Rejected customers and their reasoms.
print("\nREJECTED CUSTOMER RECORDS:")

display(
    rejected_customers[
        ["customer_id", "customer_name", "email", "age", "reject_reason"]
    ]
)

In [ ]:
# verify that every input record is accounted for

assert len(customers_test) == (
    len(valid_customers)+len(rejected_customers))
print("Customer reconciliation PASSED!")

In [ ]:

def validate_transactions(df, valid_customer_df):
    """
    Validate transaction records.

    Checks:
    - Missing transaction IDs
    - Duplicate transaction IDs
    - Missing customer IDs
    - Invalid customer references
    - Invalid transaction dates
    - Missing or negative amounts
    - Unexpected transaction statuses

    Returns:
        valid_records
        rejected_records
    """

    # Create a copy to protect the original dataset.
    data = df.copy()

    # Store rejection reasons for every row.
    rejection_reasons = [[] for _ in range(len(data))]

    # Collect all valid customer IDs.
    valid_customer_ids = set(
        valid_customer_df["customer_id"].dropna()
    )

    # Rule 1: Transaction ID cannot be missing.
    invalid_id = data["transaction_id"].isna()

    # Rule 2: Transaction ID must be unique.
    duplicate_id = (
        data["transaction_id"].notna()
        & data["transaction_id"].duplicated(keep=False)
    )

    # Rule 3: Customer ID cannot be missing.
    missing_customer = data["customer_id"].isna()

    # Rule 4: Customer ID must exist in customer dataset.
    invalid_customer = (
        data["customer_id"].notna()
        & ~data["customer_id"].isin(valid_customer_ids)
    )

    # Rule 5: Transaction date must be valid.
    invalid_date = pd.to_datetime(
        data["transaction_date"],
        errors="coerce"
    ).isna()

    # Rule 6: Amount must be present and greater than zero.
    invalid_amount = (
        data["amount"].isna()
        | data["amount"].le(0)
    )

    # Rule 7: Status must be one of the permitted values.
    allowed_statuses = {"SUCCESS", "PENDING", "FAILED"}

    invalid_status = (
        data["status"].isna()
        | ~data["status"].isin(allowed_statuses)
    )

    # Collect all applicable rejection reasons.
    for i in range(len(data)):

        if invalid_id.iloc[i]:
            rejection_reasons[i].append("Missing transaction_id")

        if duplicate_id.iloc[i]:
            rejection_reasons[i].append("Duplicate transaction_id")

        if missing_customer.iloc[i]:
            rejection_reasons[i].append("Missing customer_id")

        if invalid_customer.iloc[i]:
            rejection_reasons[i].append("Customer does not exist")

        if invalid_date.iloc[i]:
            rejection_reasons[i].append("Invalid transaction_date")

        if invalid_amount.iloc[i]:
            rejection_reasons[i].append("Amount must be greater than zero")

        if invalid_status.iloc[i]:
            rejection_reasons[i].append("Invalid transaction status")

    # Identify rejected rows.
    rejected_mask = pd.Series(
        [len(reasons) > 0 for reasons in rejection_reasons],
        index=data.index
    )

    # Separate valid and rejected records.
    valid_records = data.loc[~rejected_mask].copy()

    rejected_records = data.loc[rejected_mask].copy()

    # Attach rejection reasons.
    rejected_records["reject_reason"] = [
        "; ".join(rejection_reasons[i])
        for i in range(len(data))
        if len(rejection_reasons[i]) > 0
    ]

    return valid_records, rejected_records


print("Transaction validation function created successfully!")


In [ ]:

# Validate transaction records.
valid_transactions, rejected_transactions = validate_transactions(
    transactions_test,
    valid_customers
)

# Display the results.
print("TRANSACTION VALIDATION RESULTS")
print("------------------------------")

print("Total input records:", len(transactions_test))
print("Valid transactions:", len(valid_transactions))
print("Rejected transactions:", len(rejected_transactions))

# Display rejected transactions and their reasons.
display(
    rejected_transactions[
        [
            "transaction_id",
            "customer_id",
            "amount",
            "status",
            "reject_reason"
        ]
    ]
)


In [ ]:

# Confirm that every transaction is accounted for.

assert len(transactions_test) == (
    len(valid_transactions) + len(rejected_transactions)
)

print("Transaction reconciliation PASSED!")

# Confirm that no invalid customer references remain.
assert valid_transactions["customer_id"].isin(
    set(valid_customers["customer_id"])
).all()

print("Transaction referential integrity PASSED!")


In [ ]:
# import UUID to generate a unique run identifier

import uuid

# import date time utilities for audit timestamps
from datetime import datetime, timezone

# genrate unique id for pipeline execution
run_id = str(uuid.uuid4())

# record when this run started
run_start_time = datetime.now(timezone.utc)
print("Pipeline RunID:", run_id)
print("Run Start Time:", run_start_time.isoformat())

In [ ]:
# create reusable quarantine-saving function

def save_to_quarantine(
    rejected_df, dataset_name, run_id, source_file ):
    """
    Save rejected records with tracebility information.

    Adds:
    - run_id
    - source_file
    - rejection_timestamp
    """
    # Make a copy so we don't modify the original rejected DataFrame.
    quarantine_df = rejected_df.copy()

    # Attach the pipeline execution ID.
    quarantine_df["run_id"] = run_id

    # Record the original source filename.
    quarantine_df["source_file"] = source_file

    # Record when these records were quarantined.
    quarantine_df["rejection_timestamp"] = (
        datetime.now(timezone.utc).isoformat()
    )

    # Define the output filename.
    output_file = (
        PROJECT_ROOT
        / "data"
        / "quarantine"
        / f"rejected_{dataset_name}.csv"
    )

    # Save the rejected records.
    quarantine_df.to_csv(output_file, index=False)

    print(f"{dataset_name} quarantine saved successfully!")
    print("Rejected records:", len(quarantine_df))
    print("File location:", output_file)

    return quarantine_df


print("Quarantine function created successfully!")

    

In [ ]:
# save rejected customer records
quarantined_customers = save_to_quarantine(
    rejected_customers,
    "customers", run_id, "customers.csv")

In [ ]:
# save rejected transactions records
quarantined_transactions = save_to_quarantine(
    rejected_transactions, "transactions", run_id, "transactions_csv")

In [ ]:
# Check the files saved in the quarantine directory.
QUARANTINE_DIR = PROJECT_ROOT / "data" / "quarantine"

for file in QUARANTINE_DIR.iterdir():
    print(file.name, "-", file.stat().st_size, "bytes")

# Display a few quarantined transactions.
display(
    quarantined_transactions.head()
)


In [ ]:
# Create the processed data folder.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save only records that passed validation.
valid_customers.to_csv(
    PROCESSED_DIR / "valid_customers.csv",
    index=False
)

valid_transactions.to_csv(
    PROCESSED_DIR / "valid_transactions.csv",
    index=False
)

print("Valid records saved successfully!")
print("Valid customers:", len(valid_customers))
print("Valid transactions:", len(valid_transactions))

In [ ]:
# Reload the saved valid records to verify the files.
valid_customers_check = pd.read_csv(
    PROCESSED_DIR / "valid_customers.csv"
)

valid_transactions_check = pd.read_csv(
    PROCESSED_DIR / "valid_transactions.csv"
)

print("Customers saved:", len(valid_customers_check))
print("Transactions saved:", len(valid_transactions_check))

# Check that valid + rejected equals the original input.
assert len(valid_customers) + len(rejected_customers) == len(customers_source)
assert len(valid_transactions) + len(rejected_transactions) == len(transactions_source)

print("\nData reconciliation passed!")

In [ ]:
import hashlib
import pandas as pd

def mask_email(email):
    """
    Hide most of the email username while keeping the domain visible.
    Example: krishna@gmail.com -> k******@gmail.com
    """
    if pd.isna(email):
        return None

    email = str(email)

    if "@" not in email:
        return "INVALID_EMAIL"

    username, domain = email.split("@", 1)

    if not username:
        return "******@" + domain

    return username[0] + "*" * 6 + "@" + domain


def mask_phone(phone):
    """
    Hide all phone digits except the last four.
    """
    if pd.isna(phone):
        return None

    phone = str(phone)

    if len(phone) <= 4:
        return "*" * len(phone)

    return "*" * (len(phone) - 4) + phone[-4:]


def tokenize(value):
    """
    Generate a deterministic SHA-256 token.
    The same input always produces the same token.
    """
    if pd.isna(value):
        return None

    value = str(value)

    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [ ]:
print("Email masking:")
print(mask_email("krishna@gmail.com"))

print("\nPhone masking:")
print(mask_phone("9876543210"))

print("\nCustomer tokenization:")
print(tokenize("krishna@gmail.com"))

print("\nConsistency check:")
print(tokenize("krishna@gmail.com") == tokenize("krishna@gmail.com"))

In [ ]:
# Create a copy of validated customer records.
privacy_customers = valid_customers.copy()

# Replace direct customer identifiers with deterministic tokens.
privacy_customers["customer_token"] = (
    privacy_customers["customer_id"].apply(tokenize)
)

# Mask email and phone information.
privacy_customers["email_masked"] = (
    privacy_customers["email"].apply(mask_email)
)

privacy_customers["phone_masked"] = (
    privacy_customers["phone"].apply(mask_phone)
)

# Mask customer names to avoid exposing the original names.
privacy_customers["customer_name_masked"] = (
    privacy_customers["customer_name"].apply(
        lambda name: (
            str(name)[0] + "******"
            if pd.notna(name) and str(name)
            else None
        )
    )
)

# Display a preview of the transformed data.
display(
    privacy_customers[
        [
            "customer_id",
            "customer_token",
            "customer_name_masked",
            "email_masked",
            "phone_masked"
        ]
    ].head()
)

In [ ]:
# Confirm that the token column has no missing values.
assert privacy_customers["customer_token"].notna().all()

# Confirm that every customer has a masked email and phone.
assert privacy_customers["email_masked"].notna().all()
assert privacy_customers["phone_masked"].notna().all()

# Confirm that the original data has not been modified.
assert "email" in valid_customers.columns
assert "phone" in valid_customers.columns

print("Privacy transformation checks passed!")

In [ ]:
# Select only the customer information needed for analytics.
customer_analytics = privacy_customers[
    [
        "customer_id",
        "customer_token",
        "city",
        "age"
    ]
].copy()

# Join valid transactions with the transformed customer information.
analytical_dataset = valid_transactions.merge(
    customer_analytics,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Remove the original customer ID after joining.
analytical_dataset = analytical_dataset.drop(
    columns=["customer_id"]
)

# Display the final analytical dataset preview.
display(analytical_dataset.head())

In [ ]:
# These columns must never appear in the published analytical dataset.
sensitive_columns = {
    "customer_name",
    "email",
    "phone",
    "address",
    "customer_id"
}

# Check whether any sensitive columns remain.
exposed_columns = sensitive_columns.intersection(
    analytical_dataset.columns
)

print("Sensitive columns found:", exposed_columns)

# Stop the pipeline if any prohibited columns remain.
assert len(exposed_columns) == 0

# Ensure every transaction has a customer token.
assert analytical_dataset["customer_token"].notna().all()

print("Data minimization checks passed!")

In [ ]:
# Save the minimized dataset for downstream analytics.
analytical_dataset.to_csv(
    PROCESSED_DIR / "analytical_dataset.csv",
    index=False
)

print("Analytical dataset saved successfully!")
print("Published-ready rows:", len(analytical_dataset))
print("Published-ready columns:", len(analytical_dataset.columns))

In [ ]:
# Find transactions that do not have a matching customer token.
unmatched_transactions = analytical_dataset[
    analytical_dataset["customer_token"].isna()
]

print("Total analytical records:", len(analytical_dataset))
print("Records without customer token:", len(unmatched_transactions))

display(unmatched_transactions.head(10))

In [ ]:
# Find customer IDs in transactions that are missing from valid customers.
missing_customer_ids = set(valid_transactions["customer_id"]) - set(
    valid_customers["customer_id"]
)

print("Customer IDs without valid customer records:")
print(missing_customer_ids)

In [ ]:
# Print the exact column names.
print("Analytical dataset columns:")

for column in analytical_dataset.columns:
    print(repr(column))

# Check for duplicate column names.
print("\nDuplicate column names:")
print(analytical_dataset.columns[
    analytical_dataset.columns.duplicated()
].tolist())

In [ ]:
# Compare the original amount column with the suspicious column.
print("Original amount:")
display(analytical_dataset["amount"].head(10))

print("Suspicious amount' column:")
display(analytical_dataset["amount'"].head(10))

# Check whether both columns contain identical values.
print(
    "Are both columns identical?",
    analytical_dataset["amount"].equals(analytical_dataset["amount'"])
)

In [ ]:
# Remove the accidental column containing only missing values.
analytical_dataset = analytical_dataset.drop(
    columns=["amount'"]
)

# Verify the cleaned column names.
print("Cleaned columns:")
print(analytical_dataset.columns.tolist())

# Verify that the correct amount column is intact.
assert analytical_dataset["amount"].notna().all()

# Verify that the unwanted column is gone.
assert "amount'" not in analytical_dataset.columns

print("\nAnalytical dataset cleanup successful!")

In [ ]:
# Save the corrected analytical dataset.
analytical_dataset.to_csv(
    PROCESSED_DIR / "analytical_dataset.csv",
    index=False
)

print("Final rows:", len(analytical_dataset))
print("Final columns:", len(analytical_dataset.columns))

In [ ]:
import sqlite3

# Define the database folder and database file.
WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = WAREHOUSE_DIR / "privacy_analytics.db"

# Connect to SQLite. It creates the database if it does not exist.
connection = sqlite3.connect(DB_PATH)

# Enable foreign-key enforcement.
connection.execute("PRAGMA foreign_keys = ON")

print("Database connection successful!")
print("Database location:", DB_PATH)

In [ ]:
# Create target table with exclicit data types and constraints

create_table_sql = """
CREATE TABLE IF NOT EXISTS analytical_transactions (
    transaction_id INTEGER PRIMARY KEY,
    transaction_date TEXT NOT NULL,
    amount REAL NOT NULL CHECK (amount > 0),
    status TEXT NOT NULL
        CHECK (status IN ('SUCCESS', 'PENDING', 'FAILED')),
    customer_token TEXT NOT NULL,
    city TEXT,
    age INTEGER CHECK (age BETWEEN 18 AND 100)
);
"""

connection.execute(create_table_sql)
connection.commit()

print("Analytical target table created sucessfully!")

In [ ]:
# Select the exact columns expected by the target database.
publish_columns = [
    "transaction_id",
    "transaction_date",
    "amount",
    "status",
    "customer_token",
    "city",
    "age"
]

publish_df = analytical_dataset[publish_columns].copy()

# Convert transaction dates to ISO-8601 strings for storage.
publish_df["transaction_date"] = pd.to_datetime(
    publish_df["transaction_date"],
    errors="raise"
).apply(lambda value: value.isoformat())

# Insert records into the SQLite database.
insert_sql = """
INSERT INTO analytical_transactions (
    transaction_id,
    transaction_date,
    amount,
    status,
    customer_token,
    city,
    age
)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

records = list(
    publish_df.itertuples(index=False, name=None)
)

connection.executemany(insert_sql, records)
connection.commit()

print("Data published successfully!")
print("Records submitted:", len(records))

In [ ]:
# Count the records stored in the database.
database_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM analytical_transactions",
    connection
)

display(database_count)

# Display five published records.
published_preview = pd.read_sql_query(
    "SELECT * FROM analytical_transactions LIMIT 5",
    connection
)

display(published_preview)

# Confirm source and target counts match.
stored_count = database_count.loc[0, "total_records"]

assert stored_count == len(analytical_dataset)

print("Source-to-target reconciliation passed!")

In [ ]:
# UPSERT: insert new records or update existing records
# when the transaction_id already exists.
upsert_sql = """
INSERT INTO analytical_transactions (
    transaction_id,
    transaction_date,
    amount,
    status,
    customer_token,
    city,
    age
)
VALUES (?, ?, ?, ?, ?, ?, ?)

ON CONFLICT(transaction_id) DO UPDATE SET
    transaction_date = excluded.transaction_date,
    amount = excluded.amount,
    status = excluded.status,
    customer_token = excluded.customer_token,
    city = excluded.city,
    age = excluded.age;
"""

# Publish the same records again.
connection.executemany(upsert_sql, records)
connection.commit()

print("Safe rerun completed!")

In [ ]:
# Count all records after the second publishing.
after_rerun = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM analytical_transactions",
    connection
)

# Check whether duplicate transaction IDs exist.
duplicate_check = pd.read_sql_query(
    """
    SELECT transaction_id, COUNT(*) AS occurrences
    FROM analytical_transactions
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
    """,
    connection
)

print("Records after rerun:", after_rerun.iloc[0]["total_records"])
print("Duplicate transaction IDs:", len(duplicate_check))

assert after_rerun.iloc[0]["total_records"] == 984
assert len(duplicate_check) == 0

print("\nIdempotency verification passed!")

In [ ]:
# Store one audit record for every pipeline execution.
create_audit_table_sql = """
CREATE TABLE IF NOT EXISTS pipeline_audit (
    run_id TEXT PRIMARY KEY,
    start_time TEXT NOT NULL,
    end_time TEXT,
    status TEXT NOT NULL,
    input_customers INTEGER,
    input_transactions INTEGER,
    valid_customers INTEGER,
    valid_transactions INTEGER,
    rejected_customers INTEGER,
    rejected_transactions INTEGER,
    published_records INTEGER,
    error_message TEXT
);
"""

connection.execute(create_audit_table_sql)
connection.commit()

print("Pipeline audit table created successfully!")

In [ ]:
# Capture the pipeline completion time.
run_end_time = datetime.now(timezone.utc)

# Prepare the audit information.
audit_record = (
    run_id,
    run_start_time.isoformat(),
    run_end_time.isoformat(),
    "SUCCESS",
    len(customers_source),
    len(transactions_source),
    len(valid_customers),
    len(valid_transactions),
    len(rejected_customers),
    len(rejected_transactions),
    len(analytical_dataset),
    None
)

# Insert the audit record.
insert_audit_sql = """
INSERT INTO pipeline_audit (
    run_id,
    start_time,
    end_time,
    status,
    input_customers,
    input_transactions,
    valid_customers,
    valid_transactions,
    rejected_customers,
    rejected_transactions,
    published_records,
    error_message
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

connection.execute(insert_audit_sql, audit_record)
connection.commit()

print("Pipeline audit record saved successfully!")

In [ ]:
# Read all recorded pipeline executions.
audit_report = pd.read_sql_query(
    "SELECT * FROM pipeline_audit ORDER BY start_time DESC",
    connection
)

display(audit_report)

In [ ]:
# TEST 1: Customer reconciliation
assert len(valid_customers) + len(rejected_customers) == len(customers_source)

# TEST 2: Transaction reconciliation
assert len(valid_transactions) + len(rejected_transactions) == len(transactions_source)

# TEST 3: No duplicate customer IDs in valid records
assert valid_customers["customer_id"].is_unique

# TEST 4: No duplicate transaction IDs in valid records
assert valid_transactions["transaction_id"].is_unique

# TEST 5: All transaction references must exist in valid customers
assert set(valid_transactions["customer_id"]).issubset(
    set(valid_customers["customer_id"])
)

print("All data quality tests passed!")

In [ ]:
# TEST 6: Sensitive columns must not appear in the analytical dataset.
sensitive_columns = {
    "customer_name",
    "email",
    "phone",
    "address",
    "customer_id"
}

assert not sensitive_columns.intersection(
    analytical_dataset.columns
)

# TEST 7: Every analytical record must have a customer token.
assert analytical_dataset["customer_token"].notna().all()

# TEST 8: Original customer email and phone must not appear
# in the published analytical dataset.
assert "email" not in analytical_dataset.columns
assert "phone" not in analytical_dataset.columns

print("All privacy tests passed!")

In [ ]:
# TEST 9: Database count must match the analytical dataset.
database_count = connection.execute(
    "SELECT COUNT(*) FROM analytical_transactions"
).fetchone()[0]

assert database_count == len(analytical_dataset)

# TEST 10: Transaction IDs must be unique in the database.
database_duplicates = connection.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT transaction_id
        FROM analytical_transactions
        GROUP BY transaction_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

assert database_duplicates == 0

print("All database publishing tests passed!")

In [ ]:
# Create a small dataset containing deliberately invalid records.
negative_test_data = pd.DataFrame({
    "transaction_id": [9001, 9002, 9003, 9004],
    "customer_id": [1, 999999, 2, 3],
    "transaction_date": [
        "2026-11-01",
        "2026-11-02",
        "invalid-date",
        "2026-11-04"
    ],
    "amount": [500, 700, 800, -100],
    "status": ["SUCCESS", "SUCCESS", "PENDING", "INVALID"]
})

# Validate using our existing transaction validation function.
negative_valid, negative_rejected = validate_transactions(
    negative_test_data,
    valid_customers
)

print("Input records:", len(negative_test_data))
print("Valid records:", len(negative_valid))
print("Rejected records:", len(negative_rejected))

display(
    negative_rejected[
        ["transaction_id", "customer_id", "amount", "status", "reject_reason"]
    ]
)

In [ ]:
# Create a test dataset containing duplicate transaction IDs.
duplicate_test_data = pd.DataFrame({
    "transaction_id": [8001, 8001, 8002],
    "customer_id": [
        valid_customers.iloc[0]["customer_id"],
        valid_customers.iloc[1]["customer_id"],
        valid_customers.iloc[2]["customer_id"]
    ],
    "transaction_date": [
        "2026-11-01",
        "2026-11-02",
        "2026-11-03"
    ],
    "amount": [500, 700, 900],
    "status": ["SUCCESS", "PENDING", "SUCCESS"]
})

# Validate the test records.
duplicate_valid, duplicate_rejected = validate_transactions(
    duplicate_test_data,
    valid_customers
)

print("Input records:", len(duplicate_test_data))
print("Valid records:", len(duplicate_valid))
print("Rejected records:", len(duplicate_rejected))

display(duplicate_rejected)

In [ ]:
# Create an empty transaction DataFrame with the required schema.
empty_test_data = pd.DataFrame(columns=[
    "transaction_id",
    "customer_id",
    "transaction_date",
    "amount",
    "status"
])

# Run validation on the empty dataset.
empty_valid, empty_rejected = validate_transactions(
    empty_test_data,
    valid_customers
)

print("Input records:", len(empty_test_data))
print("Valid records:", len(empty_valid))
print("Rejected records:", len(empty_rejected))

# Confirm that empty input is handled safely.
assert len(empty_valid) == 0
assert len(empty_rejected) == 0

print("Empty input test passed!")

In [ ]:
# Test whether SQLite rejects a transaction with a negative amount.
try:
    connection.execute("""
        INSERT INTO analytical_transactions (
            transaction_id,
            transaction_date,
            amount,
            status,
            customer_token,
            city,
            age
        )
        VALUES (
            999999,
            '2026-11-13T10:00:00',
            -500,
            'SUCCESS',
            'test_token',
            'Hyderabad',
            25
        )
    """)

    connection.commit()

    print("Unexpected: Invalid record was inserted!")

except sqlite3.IntegrityError as error:
    connection.rollback()
    print("Database correctly rejected invalid data!")
    print("Reason:", error)

In [ ]:
# Try inserting an existing transaction ID.
try:
    connection.execute("""
        INSERT INTO analytical_transactions (
            transaction_id,
            transaction_date,
            amount,
            status,
            customer_token,
            city,
            age
        )
        VALUES (
            1001,
            '2026-11-13T10:00:00',
            500,
            'SUCCESS',
            'test_token',
            'Hyderabad',
            25
        )
    """)

    connection.commit()

    print("Unexpected: Duplicate transaction was inserted!")

except sqlite3.IntegrityError as error:
    connection.rollback()
    print("Duplicate transaction correctly rejected!")
    print("Reason:", error)

In [ ]:
import time

# Measure how long it takes to publish the analytical records.
performance_start = time.perf_counter()

connection.executemany(upsert_sql, records)
connection.commit()

performance_end = time.perf_counter()

publishing_time = performance_end - performance_start

print("Records processed:", len(records))
print("Publishing time:", round(publishing_time, 4), "seconds")

if publishing_time > 0:
    print(
        "Records per second:",
        round(len(records) / publishing_time, 2)
    )

In [ ]:
# Compare the analytical dataset count with the database count.
source_count = len(analytical_dataset)

target_count = connection.execute(
    "SELECT COUNT(*) FROM analytical_transactions"
).fetchone()[0]

# Compare total transaction amounts.
source_amount = round(analytical_dataset["amount"].sum(), 2)

target_amount = connection.execute(
    "SELECT ROUND(SUM(amount), 2) FROM analytical_transactions"
).fetchone()[0]

print("Source records:", source_count)
print("Target records:", target_count)

print("\nSource transaction amount:", source_amount)
print("Target transaction amount:", target_amount)

# Verify both counts and aggregate amounts match.
assert source_count == target_count
assert source_amount == target_amount

print("\nSource-to-target reconciliation passed!")

In [ ]:
import random
import time
from faker import Faker

# Use a separate Faker instance for reproducible scalability data.
scale_fake = Faker()
Faker.seed(101)
random.seed(101)

SCALE_CUSTOMERS = 10000
SCALE_TRANSACTIONS = 100000

scale_start = time.perf_counter()

# Generate customer records.
scale_customers = pd.DataFrame([
    {
        "customer_id": customer_id,
        "city": scale_fake.city(),
        "age": random.randint(18, 70)
    }
    for customer_id in range(1, SCALE_CUSTOMERS + 1)
])

# Generate transaction records.
scale_transactions = pd.DataFrame([
    {
        "transaction_id": transaction_id,
        "customer_id": random.randint(1, SCALE_CUSTOMERS),
        "transaction_date": scale_fake.date_time_between(
            start_date="-1y",
            end_date="now"
        ).isoformat(),
        "amount": round(random.uniform(100, 10000), 2),
        "status": random.choice(["SUCCESS", "PENDING", "FAILED"])
    }
    for transaction_id in range(100001, 100001 + SCALE_TRANSACTIONS)
])

scale_end = time.perf_counter()

print("Generated customers:", len(scale_customers))
print("Generated transactions:", len(scale_transactions))
print("Generation time:", round(scale_end - scale_start, 2), "seconds")

In [ ]:
# Check whether every transaction references an existing customer.
valid_customer_ids = set(scale_customers["customer_id"])

invalid_references = (
    ~scale_transactions["customer_id"].isin(valid_customer_ids)
).sum()

# Check transaction ID uniqueness.
duplicate_transaction_ids = (
    scale_transactions["transaction_id"].duplicated().sum()
)

# Check invalid transaction amounts.
invalid_amounts = (
    (scale_transactions["amount"].isna()) |
    (scale_transactions["amount"] <= 0)
).sum()

# Display the quality results.
print("Total customers:", len(scale_customers))
print("Total transactions:", len(scale_transactions))
print("Invalid customer references:", invalid_references)
print("Duplicate transaction IDs:", duplicate_transaction_ids)
print("Invalid transaction amounts:", invalid_amounts)

# Confirm the generated dataset passes these checks.
assert invalid_references == 0
assert duplicate_transaction_ids == 0
assert invalid_amounts == 0

print("\nLarge dataset quality checks passed!")

In [ ]:
# Save larger synthetic datasets separately from the original sample.
SCALE_DIR = PROJECT_ROOT / "data" / "raw" / "scale_test"
SCALE_DIR.mkdir(parents=True, exist_ok=True)

scale_customers.to_csv(
    SCALE_DIR / "scale_customers.csv",
    index=False
)

scale_transactions.to_csv(
    SCALE_DIR / "scale_transactions.csv",
    index=False
)

print("Scalability datasets saved successfully!")

In [ ]:
# Create a separate customer analytics dataset.
scale_customer_analytics = scale_customers.copy()

# Generate a stable token for each customer ID.
scale_customer_analytics["customer_token"] = (
    scale_customer_analytics["customer_id"].apply(tokenize)
)

# Keep only the fields required for analytics.
scale_customer_analytics = scale_customer_analytics[
    ["customer_id", "customer_token", "city", "age"]
]

# Join transactions with their corresponding customer information.
scale_analytical_dataset = scale_transactions.merge(
    scale_customer_analytics,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Remove the original customer ID.
scale_analytical_dataset = scale_analytical_dataset.drop(
    columns=["customer_id"]
)

print("Large analytical dataset created!")
print("Total records:", len(scale_analytical_dataset))
print("Total columns:", len(scale_analytical_dataset.columns))

display(scale_analytical_dataset.head())

In [ ]:
# Check that every transaction has a customer token.
assert scale_analytical_dataset["customer_token"].notna().all()

# Check that the original customer ID is removed.
assert "customer_id" not in scale_analytical_dataset.columns

# Check that the row count is unchanged.
assert len(scale_analytical_dataset) == len(scale_transactions)

# Check transaction ID uniqueness.
assert scale_analytical_dataset["transaction_id"].is_unique

print("Large dataset privacy checks passed!")
print("Records ready for publishing:", len(scale_analytical_dataset))

In [ ]:
# Create a separate table for the large scalability test.
connection.execute("""
CREATE TABLE IF NOT EXISTS scale_analytical_transactions (
    transaction_id INTEGER PRIMARY KEY,
    transaction_date TEXT NOT NULL,
    amount REAL NOT NULL CHECK (amount > 0),
    status TEXT NOT NULL
        CHECK (status IN ('SUCCESS', 'PENDING', 'FAILED')),
    customer_token TEXT NOT NULL,
    city TEXT,
    age INTEGER CHECK (age BETWEEN 18 AND 100)
)
""")

connection.commit()

print("Scalability target table created!")

In [ ]:
# Select the exact target columns.
scale_publish_df = scale_analytical_dataset[
    [
        "transaction_id",
        "transaction_date",
        "amount",
        "status",
        "customer_token",
        "city",
        "age"
    ]
].copy()

# Convert timestamps into ISO-8601 strings.
scale_publish_df["transaction_date"] = pd.to_datetime(
    scale_publish_df["transaction_date"],
    errors="raise"
).apply(lambda value: value.isoformat())

# Convert DataFrame rows into SQLite-compatible tuples.
scale_records = list(
    scale_publish_df.itertuples(index=False, name=None)
)

print("Records prepared:", len(scale_records))

In [ ]:
# Measure the time taken to insert 100,000 records.
scale_publish_start = time.perf_counter()

connection.executemany("""
INSERT INTO scale_analytical_transactions (
    transaction_id,
    transaction_date,
    amount,
    status,
    customer_token,
    city,
    age
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", scale_records)

connection.commit()

scale_publish_end = time.perf_counter()

scale_publish_time = scale_publish_end - scale_publish_start

print("Records published:", len(scale_records))
print("Publishing time:", round(scale_publish_time, 2), "seconds")
print(
    "Records per second:",
    round(len(scale_records) / scale_publish_time, 2)
)

In [ ]:
# Check the number of records stored in the scalability table.
scale_database_count = connection.execute("""
SELECT COUNT(*)
FROM scale_analytical_transactions
""").fetchone()[0]

print("Expected records:", len(scale_analytical_dataset))
print("Database records:", scale_database_count)

assert scale_database_count == 100000

print("100,000-record publishing verification passed!")

In [ ]:
# 1. Calculate totals from the source analytical dataset.
source_total_amount = scale_analytical_dataset["amount"].sum()

source_status_counts = (
    scale_analytical_dataset["status"]
    .value_counts()
    .to_dict()
)

# 2. Calculate totals directly from SQLite.
database_total_amount = connection.execute("""
    SELECT SUM(amount)
    FROM scale_analytical_transactions
""").fetchone()[0]

database_status_counts = dict(
    connection.execute("""
        SELECT status, COUNT(*)
        FROM scale_analytical_transactions
        GROUP BY status
    """).fetchall()
)

# 3. Check unique transaction IDs and missing customer tokens.
database_checks = connection.execute("""
    SELECT
        COUNT(*),
        COUNT(DISTINCT transaction_id),
        SUM(CASE WHEN customer_token IS NULL OR customer_token = ''
                 THEN 1 ELSE 0 END)
    FROM scale_analytical_transactions
""").fetchone()

database_count, unique_ids, missing_tokens = database_checks

# 4. Display the reconciliation results.
print("SOURCE TOTAL AMOUNT:", round(source_total_amount, 2))
print("DATABASE TOTAL AMOUNT:", round(database_total_amount, 2))

print("\nSOURCE STATUS COUNTS:", source_status_counts)
print("DATABASE STATUS COUNTS:", database_status_counts)

print("\nDATABASE RECORDS:", database_count)
print("UNIQUE TRANSACTION IDS:", unique_ids)
print("MISSING CUSTOMER TOKENS:", missing_tokens)

# 5. Assertions: stop if any integrity check fails.
assert abs(source_total_amount - database_total_amount) < 0.01
assert source_status_counts == database_status_counts
assert database_count == 100000
assert unique_ids == 100000
assert missing_tokens == 0

print("\nALL LARGE-SCALE DATA INTEGRITY CHECKS PASSED!")

In [ ]:
# Use UPSERT so existing transaction IDs are updated,
# rather than inserted as duplicate records.

scale_upsert_sql = """
INSERT INTO scale_analytical_transactions (
    transaction_id,
    transaction_date,
    amount,
    status,
    customer_token,
    city,
    age
)
VALUES (?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(transaction_id) DO UPDATE SET
    transaction_date = excluded.transaction_date,
    amount = excluded.amount,
    status = excluded.status,
    customer_token = excluded.customer_token,
    city = excluded.city,
    age = excluded.age
"""

# Run the same 100,000 records again.
connection.executemany(scale_upsert_sql, scale_records)
connection.commit()

# Verify that rerunning did not increase the record count.
after_rerun_count = connection.execute("""
SELECT COUNT(*)
FROM scale_analytical_transactions
""").fetchone()[0]

duplicate_count = connection.execute("""
SELECT COUNT(*) FROM (
    SELECT transaction_id
    FROM scale_analytical_transactions
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

print("Records after rerun:", after_rerun_count)
print("Duplicate transaction IDs:", duplicate_count)

assert after_rerun_count == 100000
assert duplicate_count == 0

print("Large-scale idempotency test passed!")

In [ ]:
# Create a report containing the scalability test results.

scalability_report = pd.DataFrame([{
    "test_name": "Large Scale Privacy Analytics Pipeline",
    "source_records": len(scale_analytical_dataset),
    "published_records": scale_database_count,
    "publishing_time_seconds": round(scale_publish_time, 4),
    "records_per_second": round(
        len(scale_records) / scale_publish_time, 2
    ),
    "source_total_amount": round(
        scale_analytical_dataset["amount"].sum(), 2
    ),
    "database_total_amount": round(
        connection.execute("""
            SELECT SUM(amount)
            FROM scale_analytical_transactions
        """).fetchone()[0], 2
    ),
    "unique_transaction_ids": after_rerun_count - duplicate_count,
    "missing_customer_tokens": int(
        scale_analytical_dataset["customer_token"].isna().sum()
    ),
    "idempotency_status": "PASSED",
    "overall_status": "PASSED"
}])

# Save the report in the reports folder.
scalability_report.to_csv(
    PROJECT_ROOT / "reports" / "scalability_test_report.csv",
    index=False
)

display(scalability_report)

print("Scalability report saved successfully!")


In [ ]:
# Display monetary values in normal decimal format.
pd.set_option("display.float_format", "{:,.2f}".format)

display(scalability_report)

In [ ]:
# Read the audit history from SQLite.
audit_results = pd.read_sql_query(
    "SELECT * FROM pipeline_audit",
    connection
)

display(audit_results)

print("Total audit records:", len(audit_results))

In [ ]:
import uuid
from datetime import datetime, timezone

# Generate a separate run ID for the scalability test.
scale_run_id = str(uuid.uuid4())

scale_start_time = datetime.now(timezone.utc)

# Collect the large-scale test metrics.
scale_input_customers = len(scale_customers)
scale_input_transactions = len(scale_transactions)
scale_published_records = scale_database_count

scale_end_time = datetime.now(timezone.utc)

# Insert a separate audit entry.
connection.execute("""
INSERT INTO pipeline_audit (
    run_id,
    start_time,
    end_time,
    status,
    input_customers,
    input_transactions,
    valid_customers,
    valid_transactions,
    rejected_customers,
    rejected_transactions,
    published_records,
    error_message
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", (
    scale_run_id,
    scale_start_time.isoformat(),
    scale_end_time.isoformat(),
    "SUCCESS",
    scale_input_customers,
    scale_input_transactions,
    scale_input_customers,
    scale_input_transactions,
    0,
    0,
    scale_published_records,
    None
))

connection.commit()

print("Scalability audit record inserted!")
print("Scalability run ID:", scale_run_id)

In [ ]:
audit_results = pd.read_sql_query(
    "SELECT * FROM pipeline_audit",
    connection
)

display(audit_results)

print("Total audit records:", len(audit_results))

assert len(audit_results) == 2
assert (audit_results["status"] == "SUCCESS").all()

print("Audit trail verification passed!")